# UAW-AFS Frequency Points Contract

This notebook validates the planned job/worker contract for UAW-AFS frequency-point simulations without launching HFSS. It uses fake setup objects to show the two intended paths:

- no `frequency_points_ghz`: keep the template sweep, such as `LINC 3.5GHz 8.7GHz 401`;
- with `frequency_points_ghz`: replace the job sweep with a PyAEDT single-point sweep over exactly those GHz points.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from worker.schemas import HfssSweepConfig, JobRequest
from worker.setup_designer import HfssSetupDesigner

REPO_ROOT

## Current Template Sweep

The existing template contract is still the default. For BCLF, the worker template typically defines a full interpolating sweep such as `LINC 3.5GHz 8.7GHz 401`.

In [ ]:
template_sweep = HfssSweepConfig(
    sweep_type='interpolating',
    sweep_data='LINC 3.5GHz 8.7GHz 401',
)

job_default = JobRequest(
    job_id='uaw_afs_contract_default',
    template_name='bclf_template_v1',
    setup_name='Setup1',
    sweep_name='Sweep1',
    variables={'L': '7.8mm', 'L1': '8.1mm', 'L2': '7.9mm', 'W': '1.0mm'},
    output_filename='uaw_afs_contract_default.s2p',
    metrics_to_extract=['max_s11_db', 'min_s21_db'],
    submit_time='2026-06-01T00:00:00Z',
)

job_default.frequency_points_ghz is None, template_sweep

## Fake HFSS Objects

The fake objects record which PyAEDT-style method would be called by the worker. They do not solve or export anything.

In [ ]:
class FakeSweep:
    def __init__(self, name):
        self.name = name
        self.props = {}


class FakeSetup:
    def __init__(self, name='Setup1'):
        self.name = name
        self.sweeps = []

    def get_sweep(self, name=None):
        if name is None:
            return self.sweeps[0] if self.sweeps else False
        return next((sweep for sweep in self.sweeps if sweep.name == name), False)

    def delete_sweep(self, name):
        for idx, sweep in enumerate(self.sweeps):
            if sweep.name == name:
                del self.sweeps[idx]
                return True
        return False

    def _add(self, name, kind):
        sweep = FakeSweep(name)
        sweep.props['kind'] = kind
        self.sweeps.append(sweep)
        return sweep

    def create_frequency_sweep(self, start_frequency=None, stop_frequency=None, num_of_freq_points=None, name=None, sweep_type='Discrete'):
        sweep = self._add(name or 'Sweep', 'frequency_sweep')
        sweep.props.update({
            'sweep_type': sweep_type,
            'start_frequency': start_frequency,
            'stop_frequency': stop_frequency,
            'num_of_freq_points': num_of_freq_points,
        })
        return sweep

    def create_linear_step_sweep(self, start_frequency=None, stop_frequency=None, step_size=None, name=None, sweep_type='Discrete'):
        sweep = self._add(name or 'Sweep', 'linear_step_sweep')
        sweep.props.update({
            'sweep_type': sweep_type,
            'start_frequency': start_frequency,
            'stop_frequency': stop_frequency,
            'step_size': step_size,
        })
        return sweep

    def create_single_point_sweep(self, name=None, unit='GHz', freq=None, save_single_field=False, save_fields=False):
        sweep = self._add(name or 'Sweep', 'single_point_sweep')
        sweep.props.update({
            'unit': unit,
            'freq': list(freq or []),
            'save_single_field': save_single_field,
            'save_fields': save_fields,
        })
        return sweep


class FakeHfss:
    def __init__(self):
        self.setups = []

    def create_setup(self, name):
        setup = FakeSetup(name)
        self.setups.append(setup)
        return setup

## Default Path: Keep Template Sweep

In [ ]:
hfss = FakeHfss()
resolved = HfssSetupDesigner().ensure_setup(
    hfss,
    setup_name='Setup1',
    sweep_name='Sweep1',
    sweep_config=template_sweep,
)

sweep = hfss.setups[0].sweeps[0]
assert resolved.setup_name == 'Setup1'
assert sweep.props['kind'] == 'frequency_sweep'
assert sweep.props['start_frequency'] == '3.5GHz'
assert sweep.props['stop_frequency'] == '8.7GHz'
assert sweep.props['num_of_freq_points'] == 401
sweep.props

## UAW-AFS Path: Override With Specific Frequency Points

`JobRequest` normalizes the optional GHz list by sorting and removing duplicates. The worker then creates a single-point sweep over those values.

In [ ]:
job_uaw = JobRequest(
    job_id='uaw_afs_contract_points',
    template_name='bclf_template_v1',
    setup_name='Setup1',
    sweep_name='Sweep1',
    frequency_points_ghz=[5.4, 3.5, 4.8, 5.4, 8.7],
    variables={'L': '7.8mm', 'L1': '8.1mm', 'L2': '7.9mm', 'W': '1.0mm'},
    output_filename='uaw_afs_contract_points.s2p',
    metrics_to_extract=['max_s11_db', 'min_s21_db'],
    submit_time='2026-06-01T00:00:00Z',
)

hfss = FakeHfss()
HfssSetupDesigner().ensure_setup(
    hfss,
    setup_name='Setup1',
    sweep_name='Sweep1',
    sweep_config=template_sweep,
    frequency_points_ghz=job_uaw.frequency_points_ghz,
)

sweep = hfss.setups[0].sweeps[0]
assert job_uaw.frequency_points_ghz == [3.5, 4.8, 5.4, 8.7]
assert sweep.props['kind'] == 'single_point_sweep'
assert sweep.props['unit'] == 'GHz'
assert sweep.props['freq'] == [3.5, 4.8, 5.4, 8.7]
assert sweep.props['save_single_field'] is False
assert sweep.props['save_fields'] is False
sweep.props

## Takeaway

The fake contract validates the intended data flow. A later real-HFSS smoke should submit one tiny `frequency_points_ghz` job and check that the exported Touchstone file contains exactly the requested frequency rows.